# Лабораторная работа 2. Обработка признаков

## Задание
1. Выбрать один или несколько наборов данных (датасетов) для решения следующих задач. Каждая задача может быть решена на отдельном датасете, или несколько задач могут быть решены на одном датасете. Просьба не использовать датасет, на котором данная задача решалась в лекции.
2. Для выбранного датасета (датасетов) на основе материалов лекций решить следующие задачи:
    1. масштабирование признаков (не менее чем тремя способами);
    2. обработку выбросов для числовых признаков (по одному способу для удаления выбросов и для замены выбросов);
    3. обработку по крайней мере одного нестандартного признака (который не является числовым или категориальным);
    4. отбор признаков:
        - один метод из группы методов фильтрации (filter methods);
        - один метод из группы методов обертывания (wrapper methods);
        - один метод из группы методов вложений (embedded methods).

## Выполнение

In [1]:
%pip install pandas numpy seaborn matplotlib scikit-learn


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, f_regression, SequentialFeatureSelector
from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.ensemble import RandomForestRegressor

Выбрать один или несколько наборов данных (датасетов) для решения следующих задач

Выбран датасет [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data?select=train.csv).

In [3]:
df = pd.read_csv('train.csv')

Масштабирование

In [4]:
data_scaling = df[['GrLivArea']].copy()

In [5]:
scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler()
}

In [6]:
for name, scaler in scalers.items():
    data_scaling[name] = scaler.fit_transform(data_scaling[['GrLivArea']])

In [7]:
print(data_scaling.head())

   GrLivArea  StandardScaler  MinMaxScaler  RobustScaler
0       1710        0.370333      0.259231      0.380070
1       1262       -0.482512      0.174830     -0.312090
2       1786        0.515013      0.273549      0.497489
3       1717        0.383659      0.260550      0.390885
4       2198        1.299326      0.351168      1.134029


Обработка выбросов

In [8]:
col = 'LotArea'

In [9]:
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [ ]:
df_removed = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

In [11]:
df_removed

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


In [12]:
upper_limit = df[col].quantile(0.95)
df_capped = df.copy()
df_capped[col] = np.where(df_capped[col] > upper_limit, upper_limit, df_capped[col])

In [13]:
print(f"\nРазмер до удаления выбросов: {df.shape[0]}, после: {df_removed.shape[0]}")


Размер до удаления выбросов: 1460, после: 1391


Обработка нестандартного признака

In [14]:
df['YearsSinceSale'] = 2024 - df['YrSold']

In [15]:
df[['YearsSinceSale']]

,YearsSinceSale
0,16
1,17
2,16
3,18
4,16
...,...
1455,17
1456,14
1457,14
1458,14


Отбор признаков

In [16]:
X = df.select_dtypes(include=[np.number]).dropna(axis=1).drop(['Id', 'SalePrice'], axis=1)
y = df['SalePrice']

один метод из группы методов фильтрации (filter methods)

In [17]:
selector_f = SelectKBest(score_func=f_regression, k=5)
X_f = selector_f.fit_transform(X, y)
print(f"Filter Method - Выбранные признаки: {list(X.columns[selector_f.get_support()])}")

Filter Method - Выбранные признаки: ['OverallQual', 'TotalBsmtSF', 'GrLivArea', 'GarageCars', 'GarageArea']


один метод из группы методов обертывания (wrapper methods)

In [18]:
lr = LinearRegression()
sfs = SequentialFeatureSelector(lr, n_features_to_select=3, direction='forward')
sfs.fit(X.iloc[:200, :10], y[:200]) # На подмножестве для скорости
print(f"Wrapper Method - Выбранные признаки: {list(X.columns[:10][sfs.get_support()])}")

Wrapper Method - Выбранные признаки: ['LotArea', 'OverallQual', 'TotalBsmtSF']


один метод из группы методов вложений (embedded methods)

In [21]:
lasso = LassoCV(cv=5).fit(X, y)
importance = np.abs(lasso.coef_)
selected_lasso = X.columns[importance > 0]
print(f"Embedded Method (Lasso) - Количество отобранных признаков: {selected_lasso}")

Embedded Method (Lasso) - Количество отобранных признаков: Index(['LotArea', 'YearBuilt', 'YearRemodAdd', 'BsmtFinSF1', 'BsmtFinSF2',
       'BsmtUnfSF', 'TotalBsmtSF', '2ndFlrSF', 'GrLivArea', 'GarageArea',
       'WoodDeckSF', 'MiscVal'],
      dtype='str')
